# Task 8 - Promo-code generator with margin logic
For each segment (Task 7) the generator **proposes** one offer and calculates its expected margin impact.

    net per targeted customer  =  extra_orders x AOV x (margin - discount)   -   baseline_orders x AOV x discount
    break-even uplift          =  baseline_p0 x discount / (margin - discount)

* `baseline_p0` (chance to order in the next 90 days with **no** promo) is **measured** by back-testing segments on history — now using merged shopping trips (1-hour gap) so the back-test isn't inflated by split checkouts.


In [1]:
import os, sys

# Notebooks live in Jupyters/, code lives in src/  ->  always run from the project root
if os.path.basename(os.getcwd()) == "Jupyters":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
assert os.path.exists("olist.db"), f"olist.db not found in {os.getcwd()} - launch Jupyter from the project root"

import pandas as pd
from IPython.display import Image, display
from src.data_loader import get_connection

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
conn = get_connection()
from src.promo_generator import *

## 1. Assumptions (the only numbers not measured from data)

In [2]:
a = Assumptions()
print("margin_rate:", a.margin_rate, "| margin_floor:", a.margin_floor,
      "| merge_gap_hours:", a.merge_gap_hours, "| uplift scenarios:", a.uplift)
print("candidate discounts:", a.discounts)

margin_rate: 0.2 | margin_floor: 0.05 | merge_gap_hours: 1.0 | uplift scenarios: {'low': 0.005, 'base': 0.015, 'high': 0.03}
candidate discounts: {'Champions': 0.1, 'Loyal repeat': 0.1, 'High-value new': 0.1, 'At-risk high-value': 0.15, 'Recent one-timers': 0.1, 'Lapsed one-timers': 0.15}


## 2. Measured: baseline order probability per segment (90-day back-test)

In [3]:
base = baseline_reorder_rates(conn, merge_gap_hours=a.merge_gap_hours)
print("segments frozen at", base.attrs["snapshot"].date(), "-> outcomes until", base.attrs["window_end"].date())
base.round(4)

segments frozen at 2018-06-01 -> outcomes until 2018-08-30


,customers_at_snapshot,ordered_in_window,p0
segment,,,
Champions,603,23,0.0381
Loyal repeat,924,13,0.0141
High-value new,17551,95,0.0054
At-risk high-value,11409,55,0.0048
Recent one-timers,26606,158,0.0059
Lapsed one-timers,18294,75,0.0041


## 3. Proposed offers

In [4]:
offers = build_offers(conn, a)
cols = ["segment", "customers", "avg_order_value", "baseline_p0", "discount", "break_even_uplift",
        "net_per_customer_low", "net_per_customer_base", "net_per_customer_high", "recommendation", "status"]
offers[cols].round(4)

,segment,customers,avg_order_value,baseline_p0,discount,break_even_uplift,net_per_customer_low,net_per_customer_base,net_per_customer_high,recommendation,status
0,Champions,844,152.88,0.0381,0.10,0.0381,-0.5067,-0.3538,-0.1245,do not discount,not_recommended
1,Loyal repeat,1201,110.47,0.0141,0.10,0.0141,-0.1002,0.0103,0.1760,send as A/B test,pending_approval
2,High-value new,21683,261.73,0.0054,0.10,0.0054,-0.0108,0.2509,0.6435,send as A/B test,pending_approval
3,At-risk high-value,14061,277.35,0.0048,0.15,0.0145,-0.1312,0.0075,0.2155,send as A/B test,pending_approval
4,Recent one-timers,32880,55.54,0.0059,0.10,0.0059,-0.0052,0.0503,0.1336,send as A/B test,pending_approval
5,Lapsed one-timers,22689,55.85,0.0041,0.15,0.0123,-0.0204,0.0075,0.0494,send as A/B test,pending_approval


In [5]:
offers[["segment", "total_net_low", "total_net_base", "total_net_high"]].round(0)

,segment,total_net_low,total_net_base,total_net_high
0,Champions,-428.0,-299.0,-105.0
1,Loyal repeat,-120.0,12.0,211.0
2,High-value new,-234.0,5441.0,13954.0
3,At-risk high-value,-1845.0,105.0,3030.0
4,Recent one-timers,-171.0,1655.0,4394.0
5,Lapsed one-timers,-462.0,171.0,1121.0


**Reading it:** re-verify after running with corrected trip counts whether
Champions are still un-discounted (they should already order at a high enough
natural rate that a code mostly gives money away), and whether other segments
are still only proposed as A/B tests rather than blanket sends.

## 4. How much does the conclusion depend on the assumed margin?

In [6]:
margin_sensitivity(conn)

,margin 20%,margin 25%,margin 30%,margin 40%
segment,,,,
Champions,-299.0,-202.0,-105.0,88.0
Loyal repeat,12.0,112.0,211.0,410.0
High-value new,5441.0,9697.0,13954.0,22466.0
At-risk high-value,105.0,3030.0,5955.0,11804.0
Recent one-timers,1655.0,3024.0,4394.0,7133.0
Lapsed one-timers,171.0,1121.0,2072.0,3972.0


## 5. Human approval gate

In [7]:
print("codes issued before approval:", len(export_customer_codes(conn, offers, merge_gap_hours=a.merge_gap_hours)))
try:
    approve_offers(offers, ["Champions"], approver="you")
except ValueError as e:
    print("blocked ->", e)

approved = approve_offers(offers, ["High-value new", "Recent one-timers"], approver="your.name")
approved[["segment", "status", "approved_by", "approved_at"]]

codes issued before approval: 0
blocked -> 'Champions' is not recommended (net impact <= 0). Use force=True to override.


,segment,status,approved_by,approved_at
0,Champions,not_recommended,NaN,NaN
1,Loyal repeat,pending_approval,NaN,NaN
2,High-value new,approved,your.name,2026-09-24 09:07
3,At-risk high-value,pending_approval,NaN,NaN
4,Recent one-timers,approved,your.name,2026-09-24 09:07
5,Lapsed one-timers,pending_approval,NaN,NaN


## 6. Export codes (approved offers only)
Each customer gets a unique code and a suggested category from the Task 6 recommender.

In [8]:
codes = export_customer_codes(conn, approved, merge_gap_hours=a.merge_gap_hours)
print(len(codes), "codes | all unique:", codes["code"].is_unique)
codes.head()

54563 codes | all unique: True


,customer_unique_id,segment,code,discount,suggested_category,valid_until
0,0000366f3b9a7992bf8c76cfdf3221e2,High-value new,SECOND10-84B72309,0.1,bed_bath_table,2026-10-24
1,0004bd2a26a76fe21f786e4fbd80607f,High-value new,SECOND10-6D42BCA8,0.1,garden_tools,2026-10-24
2,00053a61a98854899e70ed204dd4bafe,High-value new,SECOND10-9B7CDF1D,0.1,sports_leisure,2026-10-24
3,000fbf0473c10fc1ab6f8d2d286ce20c,High-value new,SECOND10-32772BDB,0.1,toys,2026-10-24
4,001928b561575b2821c92254a2327d06,High-value new,SECOND10-E2C21465,0.1,bed_bath_table,2026-10-24


In [9]:
os.makedirs("reports", exist_ok=True)
offers.round(4).to_csv("reports/promo_offers_proposal.csv", index=False)   # the proposal - safe to share
codes.to_csv("data/promo_codes_approved.csv", index=False)                 # customer-level - data/ is gitignored
print("saved")

saved
